# 03 Constrained Decoding & Logit Bias Masking

Implementing a custom PyTorch `LogitProcessor` to enforce JSON schema constraints during token sampling.

## Part 1: PyTorch Custom LogitProcessor Implementation

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.style.use('dark_background')

class JSONGrammarLogitProcessor:
    def __init__(self, allowed_token_ids):
        self.allowed_ids = allowed_token_ids
        
    def __call__(self, logits):
        mask = torch.full_like(logits, -1e9)
        mask[self.allowed_ids] = 0.0
        return logits + mask

raw_logits = torch.tensor([4.2, 3.8, 5.1, 4.9, 3.2, 2.1])
processor = JSONGrammarLogitProcessor(allowed_token_ids=[0, 1, 5]) # Only allow JSON tokens
constrained_logits = processor(raw_logits)

print('Raw Softmax Probs:', F.softmax(raw_logits, dim=-1).numpy().round(3))
print('Constrained Softmax Probs:', F.softmax(constrained_logits, dim=-1).numpy().round(3))

## Part 2: Visualizing Constrained vs Unconstrained Sampling Probabilities

In [ ]:
tokens = ['"name"', '"age"', 'class', 'SELECT', 'WHERE', '{']
raw_p = F.softmax(raw_logits, dim=-1).numpy()
con_p = F.softmax(constrained_logits, dim=-1).numpy()

x = range(len(tokens))
plt.figure(figsize=(9, 4))
plt.bar([i - 0.2 for i in x], raw_p, width=0.4, color='#ff6b6b', label='Unconstrained')
plt.bar([i + 0.2 for i in x], con_p, width=0.4, color='#51cf66', label='Constrained (Logit Bias)')
plt.xticks(x, tokens); plt.title('Logit Bias Masking: Enforcing Valid Syntax')
plt.ylabel('Sampling Probability'); plt.legend(); plt.grid(True, axis='y'); plt.show()